In [1]:
import pandas as pd
import json
import os
import requests

from shapely.geometry import LineString
from shapely.geometry import Point
from shapely.ops import unary_union

<a id="Route-analyser"></a>
<h3 style="color:#1f77b4;">Route Analyser</h3>
<a id="Loading-Input-URLS"></a>
<h5 style="color:#1fb4a7;">Load route addresses</h5>

In [2]:
HOME_LAT = -31.863708969515947
HOME_LON = 116.04151607642865

ROUTE_CSV = r"C:\Users\thoma\Documents\balga_r_deadwalk.csv"

streets_to_find = [
    "Stedham Way",
    "Brede Place",
    "St Kilda Road",
    "Fitzroy Place",
    "Mentone Road",
    "Grinstead Way",
]

##### Load reads

In [3]:
# Load meter reads and build the street matching key
df = pd.read_csv(ROUTE_CSV)
df["street_key"] = df["street"] + " " + df["type"]

print(f"Loaded {len(df)} reads")

Loaded 151 reads


##### Get street geometry from Overpass

In [4]:
cache_file = r"C:\Users\thoma\Documents\balga_streets.json"

# Reuse street data to prevent requesting many times
if os.path.exists(cache_file):
    with open(cache_file) as file:
        data = json.load(file)

else:
    # Overpass query for the selected Balga streets
    # NOTES FOR FUTURE:
    # [out:json] returns the result as JSON
    # area finds the Balga boundary and stores it as .a
    # way(area.a) searches for roads inside that boundary
    # ["highway"] keeps only road features
    # ["name"~"..."] matches any street name joined with |
    # out geom returns each road with its coordinates
    query = f"""
    [out:json];
    area["name"="Balga"]["admin_level"]->.a;
    way(area.a)["highway"]["name"~"{"|".join(streets_to_find)}"];
    out geom;
    """

    # Send Overpass API request
    response = requests.get(
        "https://overpass-api.de/api/interpreter",
        params={"data": query},
        timeout=100,
    )

    # Convert to JSON
    data = response.json()

    with open(cache_file, "w") as file:
        json.dump(data, file)

# Show number of returned elements
print(f"{len(data['elements'])} elements")

7 elements


##### Build road segments

In [5]:
# Each Overpass element is a map feature such as a road way
# A way contains tags such as the road name and geometry with latitude and longitude points
# Each way usually represents one section of a road rather than the entire street
road_segments = []

for element in data["elements"]:
    # Keep only road way elements
    if element["type"] != "way":
        continue

    name = element.get("tags", {}).get("name", "unknown")

    # Extract the points that trace the road segment
    coords = [
        (node["lat"], node["lon"])
        for node in element["geometry"]
    ]

    # Store the road segment and its endpoints
    road_segments.append(
        {
            "name": name,
            "coords": coords,
            "start": coords[0],
            "end": coords[-1],
        }
    )

    print(f"{name}: {len(coords)} geometry points")

print(f"\nCreated {len(road_segments)} road segments")

Mentone Road: 7 geometry points
Stedham Way: 7 geometry points
Brede Place: 2 geometry points
St Kilda Road: 10 geometry points
Grinstead Way: 6 geometry points
Fitzroy Place: 4 geometry points
Grinstead Way: 3 geometry points

Created 7 road segments


##### Build street outline 

In [6]:
# Convert each road segment into a line using shapely
# Shapely uses coordinates in longitude latitude order
street_lines = []

for segment in road_segments:
    coordinates = [
        (lon, lat)
        for lat, lon in segment["coords"]
    ]

    # A line needs at least two coordinate points
    if len(coordinates) >= 2:
        street_lines.append(LineString(coordinates))

# Merge all road lines into one geometry
all_streets = unary_union(street_lines)

# Add space around the roads to create one connected street shape
buffered_streets = all_streets.buffer(0.00025)

# Extract the outside boundary of the buffered streets
outline = buffered_streets.exterior
outline_points = list(outline.coords)

# Convert coordinates back to latitude longitude order
outline_path = [
    (lat, lon)
    for lon, lat in outline_points
]

print(f"Created outline with {len(outline_points)} points")

Created outline with 361 points


##### Tag outline points by street

In [7]:
# Group all road segment coordinates by street name
# Each street may contain multiple overpass way segments
street_coordinates = {}

for segment in road_segments:
    street_name = segment["name"]

    # Convert coordinates to longitude latitude for shapely
    coordinates = [
        (lon, lat)
        for lat, lon in segment["coords"]
    ]

    # Create a coordinate list for each new street
    if street_name not in street_coordinates:
        street_coordinates[street_name] = []

    # Add this segment's coordinates to the matching street
    street_coordinates[street_name].extend(coordinates)

# Create one shapely line for each named street
street_shapes = {
    street_name: LineString(coordinates)
    for street_name, coordinates in street_coordinates.items()
    if len(coordinates) >= 2
}

# Store the nearest street name for every outline point
outline_street_tags = []

for lon, lat in outline_points:
    outline_point = Point(lon, lat)

    # Find the street shape closest to this outline point
    nearest_street = min(
        street_shapes,
        key=lambda street_name: street_shapes[street_name].distance(
            outline_point
        ),
    )

    outline_street_tags.append(nearest_street)

print(f"Tagged {len(outline_street_tags)} outline points")

Tagged 361 outline points


##### Classify points by street side

In [8]:
# Match a street key against an overpass street name
# Spaces and capital letters are removed so similar names still match
def match_street(street_key, overpass_name):
    street_key = street_key.lower().replace(" ", "")
    overpass_name = overpass_name.lower().replace(" ", "")

    return street_key in overpass_name


# Determine which side of the nearest street segment a point is on
# The result depends on the direction of the street coordinates
# Returns 1 for one side and -1 for the other side
def which_side(point_lon, point_lat, line_coordinates):
    nearest_distance = float("inf")
    nearest_segment_index = 0

    # Find the street segment with the closest midpoint
    for index in range(len(line_coordinates) - 1):
        start_lon, start_lat = line_coordinates[index]
        end_lon, end_lat = line_coordinates[index + 1]

        midpoint_lon = (start_lon + end_lon) / 2
        midpoint_lat = (start_lat + end_lat) / 2

        distance = (
            (point_lon - midpoint_lon) ** 2
            + (point_lat - midpoint_lat) ** 2
        )

        if distance < nearest_distance:
            nearest_distance = distance
            nearest_segment_index = index

    # Get the start and end of the nearest street segment
    start_lon, start_lat = line_coordinates[nearest_segment_index]
    end_lon, end_lat = line_coordinates[nearest_segment_index + 1]

    street_dx = end_lon - start_lon
    street_dy = end_lat - start_lat

    point_dx = point_lon - start_lon
    point_dy = point_lat - start_lat

    # Use the cross product to identify the side of the street
    cross_product = (
        street_dx * point_dy
        - street_dy * point_dx
    )

    return 1 if cross_product > 0 else -1

##### Tag outline points by street side

In [9]:
# Store which side of its nearest street each outline point is on
# The matched street was found in the previous tagging step
outline_sides = []

for index, (lon, lat) in enumerate(outline_points):
    # Get the street already matched to this outline point
    street_name = outline_street_tags[index]

    # Compare the point with the direction of the matched street
    side = which_side(
        lon,
        lat,
        street_coordinates[street_name],
    )

    # Store 1 or -1 for the side of the street
    outline_sides.append(side)

print(f"Tagged {len(outline_sides)} outline point sides")

Tagged 361 outline point sides


##### Tag reads by street side

In [10]:
# Store which side of the street each meter read is on
# A value of 0 means no matching overpass street was found
read_sides = []

for _, row in df.iterrows():
    matched_street = None

    # Find the overpass street matching the meter read street key
    for street_name in street_coordinates:
        if match_street(row["street_key"], street_name):
            matched_street = street_name
            break

    # Store 0 when the street cannot be matched
    if matched_street is None:
        read_sides.append(0)
        continue

    # Determine the side using the read location and street direction
    side = which_side(
        row["lon"],
        row["lat"],
        street_coordinates[matched_street],
    )

    read_sides.append(side)

# Add the calculated street side to the dataframe
df["street_side"] = read_sides

print(df["street_side"].value_counts())

street_side
 1    82
-1    69
Name: count, dtype: int64


##### Snap reads to outline path

In [11]:
# Snap each meter read to its nearest matching point on the outline
# The position is the walking distance along the outline path
# Matching prefers the same street and same side of the street
positions = []

for _, row in df.iterrows():
    read_point = Point(row["lon"], row["lat"])
    read_side = row["street_side"]

    best_position = None
    best_distance = float("inf")

    # Walk the outline while tracking the cumulative distance travelled
    cumulative_distance = 0

    for index in range(len(outline_points) - 1):
        # Only snap to outline points on the same street and side
        if (
            match_street(row["street_key"], outline_street_tags[index])
            and outline_sides[index] == read_side
        ):
            outline_point = Point(outline_points[index])
            distance = read_point.distance(outline_point)

            if distance < best_distance:
                best_distance = distance
                best_position = cumulative_distance

        # Add the length of this outline segment to the running total
        point_a = Point(outline_points[index])
        point_b = Point(outline_points[index + 1])
        cumulative_distance += point_a.distance(point_b)

    # Fallback to the same street on any side
    if best_position is None:
        cumulative_distance = 0

        for index in range(len(outline_points) - 1):
            if match_street(row["street_key"], outline_street_tags[index]):
                outline_point = Point(outline_points[index])
                distance = read_point.distance(outline_point)

                if distance < best_distance:
                    best_distance = distance
                    best_position = cumulative_distance

            point_a = Point(outline_points[index])
            point_b = Point(outline_points[index + 1])
            cumulative_distance += point_a.distance(point_b)

    # Final fallback to the nearest point anywhere on the outline
    if best_position is None:
        best_position = outline.project(read_point)

    positions.append(best_position)

# Add the outline position to the dataframe
df["path_position"] = positions

print(df["path_position"].describe())

count    151.000000
mean       0.019167
std        0.010730
min        0.000034
25%        0.010646
50%        0.020107
75%        0.027986
max        0.036415
Name: path_position, dtype: float64


##### Order reads along the outline path

In [12]:
# Sort the reads by their distance along the outline path
# This creates  the walking order around the street loop
df = df.sort_values("path_position").reset_index(drop=True)

print(df[["number", "street_key", "street_side", "path_position"]].head(10).to_string(index=False))

 number    street_key  street_side  path_position
      8 ST KILDA ROAD            1       0.000034
      1 ST KILDA ROAD           -1       0.000857
      3 ST KILDA ROAD           -1       0.001144
      3 ST KILDA ROAD           -1       0.001144
      3 ST KILDA ROAD           -1       0.001144
      3 ST KILDA ROAD           -1       0.001144
      9 ST KILDA ROAD           -1       0.001170
     11 ST KILDA ROAD           -1       0.002015
    501 ST KILDA ROAD           -1       0.002015
     19 ST KILDA ROAD           -1       0.002015
